# Whisper ASR Evaluation — Nigerian-Accented English

This notebook evaluates OpenAI's Whisper model across different model sizes for transcribing Nigerian-accented English speech.

## Why this matters

Whisper is trained on a large multilingual dataset, but the distribution of Nigerian English in that data is unknown and likely very small. Nigerian English carries strong phonological influence from Yoruba, Igbo, and Hausa — tonal languages that shape how English vowels, consonants, and stress patterns are produced.

Key challenges:
- Tonal carryover (Yoruba is a 3-tone language — high, mid, low — which affects pitch patterns in speech)
- Syllable-timed rhythm vs. the stress-timed rhythm Whisper expects
- Code-switching mid-sentence (e.g. English + Yoruba fragments)
- Local proper nouns, place names, and pidgin vocabulary


## Setup

In [ ]:
# Install dependencies
# !pip install openai-whisper
# !pip install librosa soundfile

import whisper
import librosa
import soundfile as sf
import numpy as np
import time
import os

## Load Models

We compare `tiny`, `base`, and `small` — the tradeoff is inference speed vs. accuracy, which matters for real-time or near-real-time applications.

In [ ]:
model_sizes = ['tiny', 'base', 'small']
models = {}

for size in model_sizes:
    print(f'Loading whisper-{size}...')
    models[size] = whisper.load_model(size)
    print(f'  Done. Parameters: ~{sum(p.numel() for p in models[size].parameters()) / 1e6:.0f}M')

## Audio Preprocessing

Whisper expects 16kHz mono audio. Real-world audio (from phones, recordings, live environments) often comes in at different sample rates and needs to be resampled.

In [ ]:
def load_and_preprocess_audio(filepath, target_sr=16000):
    """
    Load audio file and resample to 16kHz mono — Whisper's expected format.
    """
    audio, sr = librosa.load(filepath, sr=None, mono=True)
    
    if sr != target_sr:
        print(f'  Resampling from {sr}Hz to {target_sr}Hz...')
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
    
    duration = len(audio) / target_sr
    print(f'  Loaded: {duration:.2f}s of audio')
    return audio, target_sr


def chunk_audio(audio, sr, chunk_duration=30):
    """
    Split long audio into chunks.
    Whisper works on 30s windows natively — chunking keeps memory manageable
    and allows processing of long recordings (sermons, lectures, conversations).
    """
    chunk_size = int(chunk_duration * sr)
    chunks = [audio[i:i+chunk_size] for i in range(0, len(audio), chunk_size)]
    print(f'  Split into {len(chunks)} chunk(s) of up to {chunk_duration}s each')
    return chunks

## Transcription & Benchmarking

Run each model size on the same audio and record transcription output and inference time.

In [ ]:
def transcribe_and_benchmark(audio_path, models_dict):
    """
    Transcribe a single audio file with multiple Whisper model sizes.
    Returns a dict of results per model size.
    """
    results = {}

    for size, model in models_dict.items():
        print(f'\n--- whisper-{size} ---')
        start = time.time()
        result = model.transcribe(audio_path, language='en')
        elapsed = time.time() - start

        results[size] = {
            'text': result['text'].strip(),
            'inference_time_s': round(elapsed, 2),
            'language_detected': result.get('language', 'en')
        }

        print(f'  Time: {elapsed:.2f}s')
        print(f'  Output: {result["text"][:200]}...')

    return results


# Example usage — replace with your audio file
# audio_path = 'sample_nigerian_speech.wav'
# results = transcribe_and_benchmark(audio_path, models)


## Observations on Nigerian Speech Characteristics

After running Whisper on Nigerian-accented English and Nigerian preacher recordings, some patterns emerged:

### What Whisper handles well
- Clear, moderately-paced Nigerian English — `base` and `small` are generally accurate
- Proper English words even when delivered with Nigerian accent
- Long-form monologue (e.g. sermons, lectures) where speech is structured

### Where Whisper struggles
- **Code-switching**: When a speaker shifts mid-sentence from English to Yoruba or Pidgin, Whisper often drops the non-English segment or produces hallucinated text
- **Tonal carryover**: Yoruba is a 3-tone language (high / mid / low). Speakers carry this prosody into English, which can confuse Whisper's language ID and stress detection
- **Fast speech**: Rapid Nigerian English with reduced vowels causes more errors than slower speech
- **Proper nouns**: Nigerian place names (Ile-Ife, Abeokuta, Akure), personal names, and church/denomination names are frequently misrecognised
- **Whisper-tiny**: Too lossy for reliable use on Nigerian speech — degrades significantly on accented input

### Model size recommendation
For Nigerian-accented English in a real-time context: **whisper-base** is the practical tradeoff (speed vs accuracy). **whisper-small** is better for accuracy-critical offline tasks.

## Deepgram Comparison (External)

Deepgram's Nova model was also evaluated via API for the same audio. Key differences observed:

- Deepgram is significantly faster (cloud-hosted, optimized inference)
- For Nigerian English, Deepgram and Whisper-base perform comparably on clean audio
- Deepgram requires internet access — not viable for offline deployments
- Whisper is the better choice for local/offline use cases

**Conclusion**: For offline, desktop-native applications dealing with Nigerian speech, Whisper (base or small) is the most practical open-source option currently available.

## Next Steps

- Fine-tune Whisper on Nigerian English audio to reduce accent-related errors
- Collect and annotate a more representative Nigerian speech dataset
- Explore `whisper-medium` for accuracy-critical use cases
- Investigate language ID models to detect code-switching segments before ASR